# Cross-entropy from logits in NumPy

A classifier's last layer hands you a batch of logits, one row per example and one column
per class. Most rows are ordinary. A few come from a model that has grown overconfident, with
logits in the hundreds, and one row has every logit far below zero. You will compute the
mean cross-entropy of the batch straight from the logits, without ever forming a probability
that overflows or underflows, and then its gradient with respect to the logits, which is what
training passes back to the network.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# 8 examples, 5 classes. Rows 0 to 5 are ordinary, row 6 is overconfident (logits in the
# hundreds) and row 7 has every logit far below zero.
rng = np.random.default_rng(7)
Z = rng.normal(0.0, 2.0, size=(8, 5))
Z[6] = Z[6] * 300
Z[7] = Z[7] - 900
labels = np.array([0, 3, 1, 4, 2, 2, 1, 0])
with np.printoptions(precision=1, suppress=True):
    print(Z)

## Part 1 (coding task): Log-sum-exp of every row

Write `logsumexp_rows(Z)` for a 2D array `Z`. It returns a 1D array whose entry `i` is
$\log \sum_k e^{Z_{ik}}$, and it must be finite for any finite logits. Write it without
scipy, using the identity $\log \sum_k e^{z_k} = m + \log \sum_k e^{z_k - m}$ with
$m = \max_k z_k$ taken over each row on its own.

In [ ]:
def logsumexp_rows(Z):
    """log(sum(exp(Z[i]))) for every row i, without overflow or underflow."""
    Z = np.asarray(Z, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
small = np.array([[0.0, 1.0, 2.0], [-1.0, 0.5, 0.5]])
out = logsumexp_rows(small)
assert np.shape(out) == (2,), f"expected one value per row, shape (2,), got shape {np.shape(out)}"
assert np.allclose(out, np.log(np.exp(small).sum(axis=1))), f"wrong values on small logits: {out}"
big = logsumexp_rows(np.array([[1000.0, 0.0, 0.0]]))
assert np.isfinite(big).all(), "a logit of 1000 overflows exp: shift each row by its maximum first"
assert np.isclose(big[0], 1000.0), f"logsumexp of (1000, 0, 0) is 1000 to many digits, got {big[0]}"
mixed = logsumexp_rows(np.array([[1000.0, 0.0], [-1000.0, -1000.0]]))
assert np.isfinite(mixed).all(), "a row of very negative logits underflows to log(0): shift each row by its own maximum, not the maximum of the whole array"
assert np.allclose(mixed, [1000.0, -1000.0 + np.log(2.0)]), f"expected [1000, -1000 + log 2], got {mixed}"
assert np.isclose(logsumexp_rows(np.array([[3.0, 3.0, 3.0, 3.0]]))[0], 3.0 + np.log(4.0)), "add the row maximum back after the log"
print("Part 1 passed")

## Part 2 (coding task): Mean cross-entropy from logits

Write `cross_entropy(Z, labels)`, the mean over rows of $-\log p_{i, c_i}$, where
$\mathbf{p}_i$ is the softmax of row $i$ and $c_i$ = `labels[i]` is its true class. Never
form the probabilities: use $-\log p_{i,c} = \operatorname{logsumexp}(\mathbf{z}_i) - z_{i,c}$
with your function from Part 1.

In [ ]:
def cross_entropy(Z, labels):
    """Mean cross-entropy of the rows of Z against integer class labels."""
    Z = np.asarray(Z, dtype=float)
    labels = np.asarray(labels)
    # TODO
    raise NotImplementedError

In [ ]:
Zs = np.array([[2.0, 0.0, 0.0], [0.0, 1.0, -1.0], [0.5, 0.5, 3.0]])
ys = np.array([0, 2, 1])
P = np.exp(Zs) / np.exp(Zs).sum(axis=1, keepdims=True)
want = -np.mean(np.log(P[np.arange(3), ys]))
got = cross_entropy(Zs, ys)
assert np.isclose(got, want), f"expected {want:.6f} on three small rows, got {got}: take the mean over rows of logsumexp minus the true class's logit, one logit per row"
assert np.isclose(cross_entropy(Zs[:1], ys[:1]), np.log(1 + 2 * np.exp(-2.0))), "one row of (2, 0, 0) with label 0 should give log(1 + 2e^-2)"
assert np.isclose(cross_entropy(np.array([[1000.0, 0.0, 0.0]]), [1]), 1000.0), "a confident mistake: the loss is the gap between the largest logit and the true one, 1000, not inf"
assert np.isclose(cross_entropy(np.array([[1000.0, 0.0, 0.0]]), [0]), 0.0), "a confident correct answer has a loss of about 0"
assert np.isclose(cross_entropy(np.array([[-1000.0, -1000.0]]), [0]), np.log(2.0)), "two equal logits give log 2 however negative they are"
loss = cross_entropy(Z, labels)
assert np.isfinite(loss), "the loss of the setup batch should be finite"
print(f"mean cross-entropy of the setup batch: {loss:.3f}")
print("Part 2 passed")

## Part 3 (coding task): The gradient with respect to the logits

Write `cross_entropy_grad(Z, labels)`, the gradient of the **mean** loss from Part 2 with
respect to every entry of `Z`. It has the same shape as `Z`. For one example the gradient of
$-\log p_c$ is $\mathbf{p} - \mathbf{y}$, with $\mathbf{y}$ the one-hot label, and the mean
divides each row's gradient by the number of rows. Compute the softmax stably, by subtracting
each row's maximum before the exponential.

In [ ]:
def cross_entropy_grad(Z, labels):
    """Gradient of cross_entropy(Z, labels) with respect to Z, same shape as Z."""
    Z = np.asarray(Z, dtype=float)
    labels = np.asarray(labels)
    # TODO
    raise NotImplementedError

In [ ]:
Zt = np.array([[0.3, -1.2, 0.8], [1.5, 0.2, -0.4], [-0.7, 0.9, 0.1], [0.0, 2.0, -2.0]])
yt = np.array([2, 0, 0, 1])
G = cross_entropy_grad(Zt, yt)
assert np.shape(G) == Zt.shape, f"the gradient should have the shape of Z, {Zt.shape}, got {np.shape(G)}"
h, numeric = 1e-6, np.zeros_like(Zt)
for i in range(Zt.shape[0]):
    for k in range(Zt.shape[1]):
        Up, Dn = Zt.copy(), Zt.copy()
        Up[i, k] += h
        Dn[i, k] -= h
        numeric[i, k] = (cross_entropy(Up, yt) - cross_entropy(Dn, yt)) / (2 * h)
assert np.allclose(G, numeric, atol=1e-6), f"does not match the finite-difference gradient of the mean loss:\n{np.round(G, 5)}\nexpected\n{np.round(numeric, 5)}"
assert np.allclose(G.sum(axis=1), 0.0), "each row of the gradient should sum to 0, since p and the one-hot y both sum to 1"
Gb = cross_entropy_grad(np.array([[1000.0, 0.0, 0.0], [-1000.0, -1000.0, -1000.0]]), [1, 0])
assert np.isfinite(Gb).all(), "the softmax overflowed or underflowed: subtract each row's own maximum before the exponential"
assert np.allclose(Gb, [[0.5, -0.5, 0.0], [-1 / 3, 1 / 6, 1 / 6]]), f"wrong gradient on large logits: {Gb}"
Gs = cross_entropy_grad(Z, labels)
print(f"gradient norm of the setup batch: {np.linalg.norm(Gs):.3f}")
print(f"row 6, the confident mistake: {np.round(Gs[6] * len(Z), 3)} (times n)")
print("Part 3 passed")

## Solution

### Part 1: Log-sum-exp of every row

In [ ]:
def logsumexp_rows(Z):
    """log(sum(exp(Z[i]))) for every row i, without overflow or underflow."""
    Z = np.asarray(Z, dtype=float)
    top = Z.max(axis=1)                              # one shift per row
    return top + np.log(np.exp(Z - top[:, None]).sum(axis=1))

**Why it works.** Subtracting the row maximum $m$ multiplies every term of the sum by $e^{-m}$, and
adding $m$ after the log undoes it exactly, so the value is unchanged. After the shift every
exponent is at most $0$, which rules out overflow, and one term equals $e^0 = 1$, which keeps
the sum at 1 or more and its log finite. The shift has to be per row: with one shift for
the whole array, a row far below the global maximum has every term underflow to zero.

### Part 2: Mean cross-entropy from logits

In [ ]:
def cross_entropy(Z, labels):
    """Mean cross-entropy of the rows of Z against integer class labels."""
    Z = np.asarray(Z, dtype=float)
    labels = np.asarray(labels)
    true_logit = Z[np.arange(len(Z)), labels]        # one entry per row
    return float(np.mean(logsumexp_rows(Z) - true_logit))

**Why it works.** $-\log p_c = -\log\big(e^{z_c} / \sum_k e^{z_k}\big) = \operatorname{logsumexp}(\mathbf{z}) - z_c$,
and Part 1 computes the first term safely. The true logits are picked with paired index arrays,
`Z[np.arange(n), labels]`, which takes one entry per row. `Z[:, labels]` would take every
label's column for every row, an $n \times n$ block. Going through probabilities fails on
row 6 of the setup: its true class's probability underflows to 0 and the loss becomes
infinite, while the true loss is the gap between that row's largest logit and its true one.

### Part 3: The gradient with respect to the logits

In [ ]:
def cross_entropy_grad(Z, labels):
    """Gradient of cross_entropy(Z, labels) with respect to Z, same shape as Z."""
    Z = np.asarray(Z, dtype=float)
    labels = np.asarray(labels)
    n = len(Z)
    shifted = Z - Z.max(axis=1)[:, None]
    P = np.exp(shifted)
    P /= P.sum(axis=1)[:, None]                       # stable softmax, row by row
    P[np.arange(n), labels] -= 1.0                    # p - y
    return P / n                                      # the mean divides by n

**Why it works.** Differentiating $\operatorname{logsumexp}(\mathbf{z}) - z_c$ gives the softmax
minus the one-hot vector, so after computing the stable softmax only the true class's entry
changes, by $-1$. Every row enters the mean with weight $1/n$, so every row of the gradient is
divided by $n$, which the finite-difference check sees. The confidently wrong row 6 gets a
gradient of size about $\sqrt{2}/n$, close to $+1$ on its top class and $-1$ on its true
class, the full push the lesson says cross-entropy keeps on a confident mistake.